**Volume 1 workbooks v0.1.0, published 2026-08-19.**


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

try:
    np.set_printoptions(legacy='1.25')
except TypeError:
    pass
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', '{:,.10g}'.format)

from ada_fsaudit_bridge import load_dataset, lower_bound, sample_indices, upper_bound

salaries = load_dataset('salaries')
N = len(salaries)


In [ ]:
import numpy as np
import pandas as pd
try:
    np.set_printoptions(legacy='1.25')
except TypeError:
    pass
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', '{:,.10g}'.format)

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2, norm

ADA_PLOT_COLORS = {
    'blue': '#3c4b99',
    'red': '#c93f55',
    'lightblue': '#8fa8e8',
    'lightpink': '#e36877',
}

def head(obj, n=6):
    if hasattr(obj, 'head'):
        return obj.head(n)
    return obj[:n]

sns.set_theme(style='whitegrid')


In [ ]:
import numpy as np
import pandas as pd
try:
    np.set_printoptions(legacy='1.25')
except TypeError:
    pass
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', '{:,.10g}'.format)

import numpy as np
from math import sqrt
from scipy.stats import binom, chi2, chisquare, f, hypergeom, nct, norm, poisson, t

if 'ada_set_context' not in globals():
    def ada_set_context(exercise_ref: str) -> None:
        return None


# 2. Estimating the population mean and proportion


## Exercise 2.1 Drawing a sample


In this Chapter we will use one single data file, so you can get familiar with
estimating a mean, a proportion, and their respective confidence intervals
without learning the specifics of many different datasets. The `salaries`
dataset contains salary details of 2,222 employees of the imaginary town of
Winesburg, and is included with the `FSaudit` package. The data file indicates for each employee the Grade, Step within the
Grade, and Gross salary, as well as a unique employee ID. The field Gender is
coded with a 1 for female and a 0 for male.
The practical task is to create the sample that will be used throughout the
workshop. Keeping the dataset, population size, sample size, and seed explicit
makes the later estimates reproducible.
In order to replicate the experiment in the book, you need to use the same
random number seed as we did. The sample is drawn through the notebook
bridge, inside the same R engine that the book uses and with the same
`Rounding` sample kind, so the Python workflow selects exactly the same
employees as the R workshop and the book.
Details of the data file can be obtained by inspecting the dataset columns and
row count.


In [ ]:
display(list(salaries.columns))
N = len(salaries)
print(N)


We draw a reproducible sample of 50 units from the `salaries` dataset.


In [ ]:
ada_set_context("2.1")
n = 50
sampled_rows = sample_indices(N, n, seed=12345, sample_kind="Rounding")
sample1 = salaries.iloc[sampled_rows, :]
sample1.head(6)


After running the code, check the first rows of `sample1`. This confirms that the later calculations are based on employee-level records, not only on summary numbers.


## Exercise 2.2 Point estimation of the mean


This exercise calculates the first audit estimate from the sample: the average
gross salary. The point estimate is useful, but it does not yet tell us how
precise the estimate is.
The point estimate of the mean ($\bar{y}$) is obtained by dividing the sum of
$y$ in the sample by the sample size. In Python we use the `mean()` method for
the selected data column.


In [ ]:
y_bar = np.mean(sample1["gross"])
y_bar


We can use the sample mean as an estimate for the total payroll in the population. The estimate of the total monthly payroll is then obtained by multiplying the sample mean with the number of elements in the population $N$.


In [ ]:
N * y_bar


Read the total as an extrapolation from the sample to all employees. The next exercises add uncertainty around this point estimate so it can be interpreted for audit purposes.


## Exercise 2.3 Confidence intervals around the mean


The sample mean from Exercise 2.2 is only a point estimate. In this exercise, we quantify the uncertainty around that estimate so we can judge how precise the payroll estimate is for audit work.
We start by estimating the variation in gross salaries from the sample. This variation drives the width of the confidence interval.


In [ ]:
s = np.std(sample1["gross"], ddof=1)
s


The standard error translates that salary variation to the uncertainty in the sample mean.


In [ ]:
se = s / sqrt(n)
se


For a 95% confidence interval, the $t$-value is based on tail probabilities of 0.025 and 0.975. This value sets how much uncertainty we allow on each side of the estimate.


In [ ]:
tval = t.ppf(0.975, df=(n - 1))
tval


We multiply the $t$-value with the standard error to obtain the precision achieved. Read this value as the half-width of the interval around the sample mean.


In [ ]:
precAch = tval * se
precAch


The resulting lower and upper bounds show the range of plausible values for the population mean monthly payroll. After running the code, compare the bounds with the point estimate and ask whether the interval is narrow enough for the intended audit purpose.


In [ ]:
lowerBound = y_bar - precAch
display(lowerBound)
upperBound = y_bar + precAch
display(upperBound)


For a 99\% confidence interval, the calculations are similar but use more extreme tail probabilities. The wider interval is the price of using a higher confidence level.


In [ ]:
lowerBound = y_bar + t.ppf(0.005, df=49) * se
display(lowerBound)
upperBound = y_bar + t.ppf(0.995, df=49) * se
display(upperBound)
display(N * lowerBound)
display(N * upperBound)


Compare the 95\% and 99\% intervals. The key learning point is not only how to calculate the bounds, but how confidence level and precision move together when an auditor interprets an estimate.
Before relying on these intervals, pause and judge the normality assumption for the sample mean in this salary setting. The interval above uses a $t$ calculation for the mean; that is most credible when the sample is randomly drawn and the distribution of the sample mean is close enough to normal. Use the sample size, the shape of the `gross` salaries in `sample1`, and the fact that salary populations can be skewed to write a short conclusion: is the normality assumption strong, borderline, or weak here?
If the assumption is weak, recommend a practical response before using the interval as audit evidence. For example, you might increase the sample size, inspect the population or sample shape more directly, or move toward auxiliary-variable or stratified estimation when salary level is related to grade or another useful grouping variable.


## Exercise 2.4 Extending the sample


Suppose the initial sample does not provide the desired precision for the total payroll estimate. The practical response is to collect more evidence. To work toward the target precision of $E = 600{,}000$, we extend the sample from Exercise 2.1 from 50 to 131 sampling units.
The next code chunk recreates the sample with the same seed at the larger target size, so the extended sample remains reproducible.


In [ ]:
ada_set_context("2.4")
sampled_rows = sample_indices(N, 131, seed=12345, sample_kind="Rounding")
sample2 = salaries.iloc[sampled_rows, :]


The mean of the extended sample is:


In [ ]:
np.mean(sample2["gross"])


Compare this estimate with the original sample mean. A larger sample does not guarantee a more convenient result, but it should reduce uncertainty around the estimate. The audit question is whether the additional evidence is enough to support the required precision.


## Exercise 2.5 Finite populations


This exercise checks whether the finite size of the payroll population affects sample-size planning. The practical question is whether the desired precision is tight enough that the sample becomes large relative to the population.
We saw in Section 2.2.3 that the finiteness of the population from which we sample has a noticeable effect when the sample size is greater than 10\% of the population size. In the example of the mean monthly payroll ($n = 50$, $N = 2,222$), it can be expected that this is the case if we want the precision around the total monthly payroll to be less than 300,000.
The confidence interval incorporating the finite-population correction factor is given by Equation 2.10:


In [ ]:
E = 300000
tval = t.ppf(0.975, df=(n - 1))
gamma = E**2 / (N * tval**2 * np.std(sample1["gross"], ddof=1)**2)
display(gamma)
display((N / (1 + gamma)))


The total sample size required to obtain an estimate of the mean monthly payroll with a precision of 300,000 at 95% confidence is 424. This is more than 10% of the population size; therefore, we were right to use the finite-population correction factor. If we had not anticipated that the required sample size would have been unnecessarily large, not taking the finite-population correction factor into account would have resulted in a sample size of 524.


In [ ]:
(N**2 * tval**2) * np.std(sample1["gross"], ddof=1)**2 / E**2


Compare the corrected and uncorrected sample sizes. The audit planning point is that population size can materially change the evidence required when the planned sample is a substantial share of the population.


## Exercise 2.6 Confidence intervals around the proportion


The same sample can also be used for a qualitative characteristic. Here, the
task is to estimate the proportion of women in the payroll population using the
`Gender` field introduced in Exercise 2.1.
In the sample, `k = 31` employees are coded as female. These bounds are
calculated with the `upper_bound()` and `lower_bound()` helpers, which mirror
the corresponding confidence-bound calculations.


In [ ]:
display(lower_bound(k = 31, popn = 2222, n = 50, alpha = .025))
display(upper_bound(k = 31, popn = 2222, n = 50, alpha = .025))


These bounds use the finite population size. They answer the practical question: based on the sample, what range of population proportions is still plausible?
Binomial bounds can be obtained by adding the distribution to the respective functions. This gives a useful comparison because it treats the population differently.


In [ ]:
display(lower_bound(k=31, popn=50, n=50, alpha=.025, dist="binom") / 50)
display(upper_bound(k=31, popn=50, n=50, alpha=.025, dist="binom") / 50)
display(lower_bound(k = 31, popn = 2222, n = 50, alpha = .025, dist = "binom"))
display(upper_bound(k = 31, popn = 2222, n = 50, alpha = .025, dist = "binom"))


Compare the hypergeometric and binomial bounds. The hypergeometric distribution uses the finite population information directly, so it provides a more precise result here. The learning point is that distribution choice can affect how much uncertainty remains around a proportion estimate.
